# Week 11: AI Recommendation Engine

Build logic for Buy/Sell/Hold decisions: Combine predicted return 

Design decision rules: Example: Buy if pred_return > 2%

First version of AI investor bot working.

Outputs recommendation with justification.


# Week 12: Final Report, UI & Presentation
Tasks:
Create final project report (PDF or notebook):


Methodology, analysis, ML models, sentiment impact


Prepare slides for final presentation.


Finalize Streamlit/Gradio app (optional).


Record demo (optional) and publish repo.


Milestones:
Complete GitHub repo with code + documentation.


Report, presentation, and optional web app submitted.


In [27]:
# 1) Task 1  — Build 'pred_combined' (pred_* avg → y_pred → lagged log return)
# 2) Task 2  — Apply decision rules (BUY if pred > +2%, SELL if pred < -2%, else HOLD)
# 3) Milestone 1 — First working AI investor bot (end-to-end)
# 4) Milestone 2 — Outputs recommendation with justification (saved snapshot)

# Outputs:
# - week11/task1_combined_pred_latest.csv
# - week11/task2_recommendations.csv / .md
# - week11/ai_bot_m1_recommendations.csv / .md
# - week11/milestone2_recommendations.csv / .md

import numpy as np
import pandas as pd
from pathlib import Path
from IPython.display import display, Markdown

REPORT_DIR = Path("week10")
OUT_DIR    = Path("week11")
OUT_DIR.mkdir(parents=True, exist_ok=True)

# thresholds (log-return units; 0.002 ≈ +0.2%)
POS_THRESHOLD = 0.002
NEG_THRESHOLD = -0.002
ALLOW_SHORT   = False  # long-only; SELL => avoid/close long (no short)


def load_eval():
    """Load evaluation table (Parquet with CSV fallback), ensure Date dtype, sort."""
    pq, csv = REPORT_DIR / "eval_table.parquet", REPORT_DIR / "eval_table.csv"
    if pq.exists():
        try:
            df = pd.read_parquet(pq).copy()
        except Exception:
            if csv.exists():
                df = pd.read_csv(csv, parse_dates=["Date"]).copy()
            else:
                raise
    elif csv.exists():
        df = pd.read_csv(csv, parse_dates=["Date"]).copy()
    else:
        raise FileNotFoundError("Neither eval_table.parquet nor eval_table.csv found in week10/")
    if not pd.api.types.is_datetime64_any_dtype(df["Date"]):
        df["Date"] = pd.to_datetime(df["Date"], errors="coerce")
    return df.sort_values(["Ticker","Date"]).reset_index(drop=True)

def compute_pred_combined(df: pd.DataFrame) -> pd.Series:
    """
    Priority:
      1) mean of pred_* columns (excluding y_pred) if any
      2) else y_pred if present
      3) else naive = yesterday's log return from Close
    """
    pred_cols = [c for c in df.columns
                 if c.lower().startswith("pred_") and c != "y_pred"
                 and pd.api.types.is_numeric_dtype(df[c])]
    if pred_cols:
        return df[pred_cols].mean(axis=1)
    if "y_pred" in df.columns and pd.api.types.is_numeric_dtype(df["y_pred"]):
        return df["y_pred"]
    if "Close" in df.columns:
        logret = np.log(df["Close"] / df["Close"].shift(1))
        return logret.shift(1)
    raise ValueError("No prediction columns found and 'Close' missing; cannot build pred_combined.")

def decide_action(pred: float, pos_thr: float, neg_thr: float, allow_short: bool):
    """
    Stateless decision: returns (action, justification).
    action ∈ {"BUY","SELL","HOLD"}; with shorting OFF, SELL means avoid/close long.
    """
    if pred > pos_thr:
        return (
            "BUY",
            f"Predicted next-day log return {pred:.2%} > {pos_thr:.2%} threshold."
        )
    if pred < neg_thr:
        if allow_short:
            return (
                "SELL",
                f"Predicted next-day log return {pred:.2%} < {neg_thr:.2%}; open short."
            )
        else:
            return (
                "SELL",
                f"Predicted next-day log return {pred:.2%} < {neg_thr:.2%}; avoid/close long (no short)."
            )
    return (
        "HOLD",
        f"Predicted return within dead-zone [{neg_thr:.2%}, {pos_thr:.2%}]."
    )

# =============================================================================
# Combine predicted return
# =============================================================================
eval_df = load_eval()
eval_df["pred_combined"] = compute_pred_combined(eval_df)

task1_latest = (
    eval_df.groupby("Ticker", as_index=False)
           .tail(1)[["Ticker","Date","pred_combined"]]
           .sort_values("Ticker")
           .reset_index(drop=True)
)

display(Markdown("## Task 1 — Latest combined predicted next-day log return"))
display(task1_latest)

task1_csv = OUT_DIR / "task1_combined_pred_latest.csv"
task1_latest.to_csv(task1_csv, index=False)
print("Saved:", task1_csv)

# =============================================================================
# Design decision rules (±2% example)
# =============================================================================
task2_rows = []
for _, r in task1_latest.iterrows():
    pred = float(r["pred_combined"])
    action, reason = decide_action(pred, POS_THRESHOLD, NEG_THRESHOLD, ALLOW_SHORT)
    task2_rows.append({
        "Ticker": r["Ticker"],
        "Date":   pd.to_datetime(r["Date"]).date(),
        "pred_combined": pred,
        "Decision": action,
        "Justification": reason
    })
task2_recs = pd.DataFrame(task2_rows).sort_values("Ticker")

display(Markdown(f"## Task 2 — Recommendations (long-only)  \n"
                 f"Rule: BUY if pred > +{POS_THRESHOLD:.0%}, SELL if pred < {NEG_THRESHOLD:.0%}, else HOLD."))
display(task2_recs)

task2_csv = OUT_DIR / "task2_recommendations.csv"
task2_md  = OUT_DIR / "task2_recommendations.md"
task2_recs.to_csv(task2_csv, index=False)
md_lines = [
    "# Task 2 — Recommendations (long-only)\n\n",
    f"Rule: **BUY** if pred_combined > +{POS_THRESHOLD:.0%}, **SELL** if pred_combined < {NEG_THRESHOLD:.0%}, else **HOLD**.\n\n",
    "| Ticker | Date | pred_combined | Decision | Justification |\n",
    "|---|---|---:|---|---|\n",
]
for _, rr in task2_recs.iterrows():
    md_lines.append(f"| {rr.Ticker} | {rr.Date} | {rr.pred_combined:.5f} | {rr.Decision} | {rr.Justification} |\n")
task2_md.write_text("".join(md_lines), encoding="utf-8")
print("Saved:", task2_csv, "and", task2_md)

# =============================================================================
# First working AI investor bot (end-to-end)
# =============================================================================
m1_rows = []
for _, r in task1_latest.iterrows():
    pred = float(r["pred_combined"])
    action, reason = decide_action(pred, POS_THRESHOLD, NEG_THRESHOLD, ALLOW_SHORT)
    m1_rows.append({
        "Ticker": r["Ticker"],
        "Date":   pd.to_datetime(r["Date"]).date(),
        "pred_combined": pred,
        "Decision": action,
        "Justification": reason
    })
m1_recs = pd.DataFrame(m1_rows).sort_values("Ticker")

display(Markdown("## Milestone 1 — AI Investor Bot (2% / -2% rule, long-only)"))
display(m1_recs)

m1_csv = OUT_DIR / "ai_bot_m1_recommendations.csv"
m1_md  = OUT_DIR / "ai_bot_m1_recommendations.md"
m1_recs.to_csv(m1_csv, index=False)

md = [
    "# Milestone 1 — AI Investor Bot (2% / -2% rule, long-only)\n\n",
    f"Rule: **BUY** if pred_combined > +{POS_THRESHOLD:.0%}, **SELL** if pred_combined < {NEG_THRESHOLD:.0%}, else **HOLD**.\n\n",
    "| Ticker | Date | pred_combined | Decision | Justification |\n",
    "|---|---|---:|---|---|\n",
]
for _, rr in m1_recs.iterrows():
    md.append(f"| {rr.Ticker} | {rr.Date} | {rr.pred_combined:.5f} | {rr.Decision} | {rr.Justification} |\n")
Path(m1_md).write_text("".join(md), encoding="utf-8")
print("Saved:", m1_csv, "and", m1_md)

# =============================================================================
# Outputs recommendation with justification
# =============================================================================
m2_recs = m1_recs.copy()

display(Markdown("## Milestone 2 — Recommendations with justification (±2% rule)"))
display(m2_recs)

m2_csv = OUT_DIR / "milestone2_recommendations.csv"
m2_md  = OUT_DIR / "milestone2_recommendations.md"
m2_recs.to_csv(m2_csv, index=False)

md2 = [
    "# Milestone 2 — Recommendations with justification (±2% rule)\n\n",
    f"Rule: **BUY** if pred_combined > +{POS_THRESHOLD:.0%}, **SELL** if pred_combined < {NEG_THRESHOLD:.0%}, else **HOLD**.\n\n",
    "| Ticker | Date | pred_combined | Decision | Justification |\n",
    "|---|---|---:|---|---|\n",
]
for _, rr in m2_recs.iterrows():
    md2.append(f"| {rr.Ticker} | {rr.Date} | {rr.pred_combined:.5f} | {rr.Decision} | {rr.Justification} |\n")
Path(m2_md).write_text("".join(md2), encoding="utf-8")
print("Saved:", m2_csv, "and", m2_md)

## Task 1 — Latest combined predicted next-day log return

,Ticker,Date,pred_combined
0,AAPL,2025-07-09,0.000286
1,AMZN,2025-07-09,-0.018563
2,CRM,2025-07-09,0.014169
3,GOOGL,2025-07-09,-0.013840
4,IBM,2025-07-09,-0.007034
5,META,2025-07-09,0.003224
6,MSFT,2025-07-09,-0.002213
7,NVDA,2025-07-08,-0.006927
8,ORCL,2025-07-08,-0.021552
9,TSLA,2025-07-09,0.013080


Saved: week11/task1_combined_pred_latest.csv


## Task 2 — Recommendations (long-only)  
Rule: BUY if pred > +0%, SELL if pred < -0%, else HOLD.

,Ticker,Date,pred_combined,Decision,Justification
0,AAPL,2025-07-09,0.000286,HOLD,"Predicted return within dead-zone [-0.20%, 0.2..."
1,AMZN,2025-07-09,-0.018563,SELL,Predicted next-day log return -1.86% < -0.20%;...
2,CRM,2025-07-09,0.014169,BUY,Predicted next-day log return 1.42% > 0.20% th...
3,GOOGL,2025-07-09,-0.013840,SELL,Predicted next-day log return -1.38% < -0.20%;...
4,IBM,2025-07-09,-0.007034,SELL,Predicted next-day log return -0.70% < -0.20%;...
5,META,2025-07-09,0.003224,BUY,Predicted next-day log return 0.32% > 0.20% th...
6,MSFT,2025-07-09,-0.002213,SELL,Predicted next-day log return -0.22% < -0.20%;...
7,NVDA,2025-07-08,-0.006927,SELL,Predicted next-day log return -0.69% < -0.20%;...
8,ORCL,2025-07-08,-0.021552,SELL,Predicted next-day log return -2.16% < -0.20%;...
9,TSLA,2025-07-09,0.013080,BUY,Predicted next-day log return 1.31% > 0.20% th...


...

,Ticker,Date,pred_combined,Decision,Justification
0,AAPL,2025-07-09,0.000286,HOLD,"Predicted return within dead-zone [-0.20%, 0.2..."
1,AMZN,2025-07-09,-0.018563,SELL,Predicted next-day log return -1.86% < -0.20%;...
2,CRM,2025-07-09,0.014169,BUY,Predicted next-day log return 1.42% > 0.20% th...
3,GOOGL,2025-07-09,-0.013840,SELL,Predicted next-day log return -1.38% < -0.20%;...
4,IBM,2025-07-09,-0.007034,SELL,Predicted next-day log return -0.70% < -0.20%;...
5,META,2025-07-09,0.003224,BUY,Predicted next-day log return 0.32% > 0.20% th...
6,MSFT,2025-07-09,-0.002213,SELL,Predicted next-day log return -0.22% < -0.20%;...
7,NVDA,2025-07-08,-0.006927,SELL,Predicted next-day log return -0.69% < -0.20%;...
8,ORCL,2025-07-08,-0.021552,SELL,Predicted next-day log return -2.16% < -0.20%;...
9,TSLA,2025-07-09,0.013080,BUY,Predicted next-day log return 1.31% > 0.20% th...


Saved: week11/ai_bot_m1_recommendations.csv and week11/ai_bot_m1_recommendations.md


## Milestone 2 — Recommendations with justification (±2% rule)

,Ticker,Date,pred_combined,Decision,Justification
0,AAPL,2025-07-09,0.000286,HOLD,"Predicted return within dead-zone [-0.20%, 0.2..."
1,AMZN,2025-07-09,-0.018563,SELL,Predicted next-day log return -1.86% < -0.20%;...
2,CRM,2025-07-09,0.014169,BUY,Predicted next-day log return 1.42% > 0.20% th...
3,GOOGL,2025-07-09,-0.013840,SELL,Predicted next-day log return -1.38% < -0.20%;...
4,IBM,2025-07-09,-0.007034,SELL,Predicted next-day log return -0.70% < -0.20%;...
5,META,2025-07-09,0.003224,BUY,Predicted next-day log return 0.32% > 0.20% th...
6,MSFT,2025-07-09,-0.002213,SELL,Predicted next-day log return -0.22% < -0.20%;...
7,NVDA,2025-07-08,-0.006927,SELL,Predicted next-day log return -0.69% < -0.20%;...
8,ORCL,2025-07-08,-0.021552,SELL,Predicted next-day log return -2.16% < -0.20%;...
9,TSLA,2025-07-09,0.013080,BUY,Predicted next-day log return 1.31% > 0.20% th...


Saved: week11/milestone2_recommendations.csv and week11/milestone2_recommendations.md


In [37]:
# gradio
import io
import gradio as gr

REPORT_DIR = Path("week10")

def load_eval():
    pq, csv = REPORT_DIR / "eval_table.parquet", REPORT_DIR / "eval_table.csv"
    if pq.exists():
        try:
            df = pd.read_parquet(pq).copy()
        except Exception:
            if csv.exists():
                df = pd.read_csv(csv, parse_dates=["Date"]).copy()
            else:
                raise
    elif csv.exists():
        df = pd.read_csv(csv, parse_dates=["Date"]).copy()
    else:
        raise FileNotFoundError("Neither eval_table.parquet nor eval_table.csv found in week10/")
    if not pd.api.types.is_datetime64_any_dtype(df["Date"]):
        df["Date"] = pd.to_datetime(df["Date"], errors="coerce")
    return df.sort_values(["Ticker","Date"]).reset_index(drop=True)

def ensure_truth(df):
    if "target_next_log_return" in df.columns and pd.api.types.is_numeric_dtype(df["target_next_log_return"]):
        return df["target_next_log_return"]
    if "Close" in df.columns:
        logret = np.log(df["Close"] / df["Close"].shift(1))
        return logret.shift(-1)
    raise ValueError("Need target_next_log_return or Close for y_true")

def compute_pred_combined(df):
    pred_cols = [c for c in df.columns if c.lower().startswith("pred_") and c != "y_pred"
                 and pd.api.types.is_numeric_dtype(df[c])]
    if pred_cols: return df[pred_cols].mean(axis=1)
    if "y_pred" in df.columns and pd.api.types.is_numeric_dtype(df["y_pred"]): return df["y_pred"]
    if "Close" in df.columns:
        logret = np.log(df["Close"] / df["Close"].shift(1))
        return logret.shift(1)
    raise ValueError("No prediction columns and no Close; cannot build pred_combined.")

def decide_action(pred, pos_thr, neg_thr, allow_short):
    if pred > pos_thr:
        return ("BUY",  f"Pred {pred:.2%} > +{pos_thr:.2%}.")
    if pred < neg_thr:
        if allow_short: return ("SELL", f"Pred {pred:.2%} < {neg_thr:.2%}; open short.")
        return ("SELL", f"Pred {pred:.2%} < {neg_thr:.0%}; avoid/close long (no short).")
    return ("HOLD", f"Pred within dead-zone [{neg_thr:.0%}, +{pos_thr:.2%}].")

def run_bot(pos_thr_pct, neg_thr_pct, allow_short):
    pos_thr = pos_thr_pct / 100.0
    neg_thr = neg_thr_pct / 100.0

    df = load_eval()
    if "pred_combined" not in df.columns:
        df["pred_combined"] = compute_pred_combined(df)

    latest = df.groupby("Ticker", as_index=False).tail(1).reset_index(drop=True)

    rows = []
    for _, r in latest.iterrows():
        pred = float(r["pred_combined"])
        action, reason = decide_action(pred, pos_thr, neg_thr, allow_short)
        rows.append({
            "Ticker": r["Ticker"],
            "Date":   pd.to_datetime(r["Date"]).date(),
            "pred_combined": pred,
            "Decision": action,
            "Justification": reason
        })
    recs = pd.DataFrame(rows).sort_values("Ticker")

    csv_buf = io.StringIO()
    recs.to_csv(csv_buf, index=False)
    return recs, csv_buf.getvalue()

with gr.Blocks(title="AI Investor Bot") as demo:
    gr.Markdown("# AI Investor Bot — Quick Recommendations")
    with gr.Row():
        pos_thr = gr.Slider(0.0, 3.0, value=0.2, step=0.05, label="BUY threshold (+%)")
        neg_thr = gr.Slider(-3.0, 0.0, value=-0.2, step=0.05, label="SELL threshold (−%)")
        allow_s = gr.Checkbox(label="Allow shorting", value=False)
    run_btn = gr.Button("Run")
    out_tbl = gr.Dataframe(headers=["Ticker","Date","pred_combined","Decision","Justification"], wrap=True)
    out_csv = gr.File(label="Download CSV", visible=False)

    def _on_run(p, n, s):
        df, csv_text = run_bot(p, n, s)
        tmp = Path("latest_orders.csv")
        df.to_csv(tmp, index=False)
        return df, str(tmp)

    run_btn.click(_on_run, inputs=[pos_thr, neg_thr, allow_s], outputs=[out_tbl, out_csv])

if __name__ == "__main__":
    demo.launch()

* Running on local URL:  http://127.0.0.1:7880
* To create a public link, set `share=True` in `launch()`.
